## Notebook to download human proteome AF2 model and run AFragmenter

### Step 1 - Get uniprot accession from UniProt human proteome (UP000005640)
- AFragmenter CLI entry point requires uniprot accession as an argument to download the latest AF2 model.
- Get uniprot human proteome from this URL, then break uniprot accessions down into subset, which will be read by SLURM array jobs

URL for human proteome:
https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29


In [ ]:
import datetime
import subprocess

# Download human proteome .tsv file
proteome_url = "https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29"
access_date = datetime.datetime.today().strftime('%Y_%m_%d')
proteome_path = f"./data/proteome/uniprotkb_proteome_UP000005640_{access_date}.tsv.gz"

# Use subprocess to call wget properly with arguments as a list
subprocess.run([
    "wget", proteome_url, "-O", proteome_path
], check=True)

# Unzip the downloaded file
subprocess.run([
    "gunzip", proteome_path
], check=True)

proteome_path = proteome_path.replace('.gz', '')

--2025-11-27 09:44:08--  https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29
Resolving rest.uniprot.org (rest.uniprot.org)... 193.62.193.81
Connecting to rest.uniprot.org (rest.uniprot.org)|193.62.193.81|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [text/plain]
Saving to: ‘./data/proteome/uniprotkb_proteome_UP000005640_2025_11_27.tsv’

     0K .......... .......... .......... .......... .......... 94.4K
    50K .......... .......... .......... .......... .......... 82.0K
   100K .......... .......... .......... .......... ..........  160K
   150K .......... .......... .......... .......... .......... 56.0K
   200K .......... .......... .......... .......... .......... 85.7K
   250K .......... .......... .......... .......... ..........  155K
   300K .......... .......... .......... .......... ..........  1

CompletedProcess(args=['wget', 'https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Creviewed%2Cid%2Cprotein_name%2Cgene_names%2Corganism_name%2Clength&format=tsv&query=%28%28proteome%3AUP000005640%29%29', '-O', './data/proteome/uniprotkb_proteome_UP000005640_2025_11_27.tsv'], returncode=0)

In [5]:
# Try to read uniref_identity_0_5_AND_taxonomy_id_2025_11_26.tsv
import pandas as pd

df_proteome = pd.read_csv(proteome_path, sep='\t')

df_proteome.head()


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length
0,A0A087WVL8,unreviewed,A0A087WVL8_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),548
1,A0A087WXI3,unreviewed,A0A087WXI3_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),536
2,A0A087WY29,unreviewed,A0A087WY29_HUMAN,Fragile X messenger ribonucleoprotein 1 (Fragi...,FMR1,Homo sapiens (Human),561
3,A0A087WYG2,unreviewed,A0A087WYG2_HUMAN,C-Jun-amino-terminal kinase-interacting protei...,MAPK8IP3,Homo sapiens (Human),1337
4,A0A087WZT3,unreviewed,A0A087WZT3_HUMAN,BOLA2-SMG1P6 readthrough,BOLA2-SMG1P6,Homo sapiens (Human),44


In [7]:
# split all rows of df_proteome into 500 subset dfs, and save each subset to a .csv file to data/subset
import os

subset_dir = "data/subset"
os.makedirs(subset_dir, exist_ok=True)

num_subsets = 500
for i in range(num_subsets):
    subset_df = df_proteome.iloc[i::num_subsets]
    subset_df.to_csv(f'data/subset/subset_{i}.csv', index=False)


Run afdb_to_domain.sh slurm job on SLURM

In [9]:
# Submit the job to the cluster
# subset_dir = "data/subset"       # dir containing input subset csv files
out_dir = "data/out"               # dir to save domainome .pdb and domains .csv files
domainome_dir = "data/domainome"   # dir to save subset out csv files (for tracking status of each subset)
!sbatch  --array 0 slurm/afdb_to_domain.sh $subset_dir $domainome_dir $out_dir

sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 1.75        │ 0.05        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 1.65        │ 0.05        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)
Submitted batch job 43771778


### Step 2 - summarize results, and compute postprocessing metrics